In [1]:
using LowLevelFEM

# Stationary Stokes Flow

This example demonstrates the solution of the stationary incompressible
Stokes equations using the LowLevelFEM weak-form DSL.

The coupled velocity–pressure system is assembled from individual weak
forms and combined into a block system matrix.

In [2]:
structured_rect_mesh(lx=2, order=3)

## Geometry

A two-dimensional rectangular domain is discretized using a structured
quadratic finite element mesh.

## Unknown fields

Two independent finite element fields are introduced:

- **v**: velocity vector field,
- **p**: pressure scalar field.

The coupled Stokes system is assembled from these two fields.

In [3]:
material = Material("body")

Pp = Problem([material], type=:ScalarField, field=:p, rhs_field=:fp, dim=2)
Pv = Problem([material], type=:VectorField, field=:v, rhs_field=:fv, dim=2);

## Boundary conditions

The pressure is fixed at one corner to remove the pressure null space.

The velocity is prescribed to be zero on the upper and lower boundaries
(no-slip condition).

In [4]:
pres1 = BoundaryCondition("rightbottom", problem=Pp, p=0);

In [5]:
suppT = BoundaryCondition("top", problem=Pv, vx=0, vy=0)
suppB = BoundaryCondition("bottom", problem=Pv, vx=0, vy=0);

## Body force

A constant horizontal body force drives the flow throughout the domain.

In [6]:
load_v = LoadCondition("body", fvx=1.0, fvy=0.0)
fv = loadVector(Pv, [load_v])

gp = loadVector(Pp, [])

F = SystemVector([fv, gp]);

## Weak formulation

The Stokes equations are assembled from individual bilinear forms:

- viscous term,
- grad-div stabilization,
- velocity–pressure coupling,
- pressure Laplacian stabilization.

The individual operators are later combined into a single block system.

In [7]:
μ = 1.0

A = ∫((SymGrad(Pv) ⋅ SymGrad(Pv)) * 2μ)

B = ∫(Div(Pv) ⋅ Pp);

### Stabilization

Two stabilization terms are included:

- **γ**: grad-div stabilization, improving mass conservation,
- **δ**: pressure Laplacian stabilization, suppressing spurious pressure
  oscillations.

The pressure stabilization parameter depends on the mesh size.

In [8]:
γ = 1e-1          # grad-div ( 1e-2...1e0)
δ = 1e-4          # pressure Laplacian (mesh dependent)

C = ∫(Grad(Pp) ⋅ Grad(Pp) * δ)

D = ∫(Div(Pv) ⋅ Div(Pv) * γ);

### Combined weak form

The viscous and grad-div contributions can either be assembled separately

```julia
A + D
```

or directly as a single weak form

```julia
AD = ∫(...)
```

Both approaches produce identical system matrices.

In [9]:
# alternative way
AD = ∫((SymGrad(Pv) ⋅ SymGrad(Pv)) * 2μ + γ * (Div(Pv) ⋅ Div(Pv)));

### Modular assembly

Each physical contribution is assembled independently and can be added,
removed, or modified without changing the remaining terms.

This makes it straightforward to experiment with alternative
stabilization strategies or constitutive models.

## Block system

The global saddle-point system is assembled as

$$
\begin{bmatrix}
A+D & B \\
B^T & -C
\end{bmatrix},
$$

where

- **A** – viscous operator,
- **D** – grad-div stabilization,
- **B** – velocity–pressure coupling,
- **C** – pressure stabilization.

In [10]:
K = SystemMatrix([A+D B;
    B' -C])

K[:, :]

5673×5673 SparseArrays.SparseMatrixCSC{Float64, Int64} with 409025 stored entries:
⎡⡿⣯⣛⣛⣻⡇⡃⡃⡛⡛⣘⣘⢘⢘⢘⢃⢃⡃⡃⡃⡇⡶⣰⣰⢸⢼⣽⣟⣻⣟⣛⣛⣛⣛⣛⣛⣻⣶⣾⣯⎤
⎢⣿⢸⡿⣯⡁⠛⠷⠷⠧⠥⣭⣭⣬⣈⣈⣈⣈⡁⠁⠁⠀⠀⠀⠀⠀⠀⢸⣿⣏⠿⠿⢭⣭⣉⣉⡉⠁⠀⠀⠀⎥
⎢⠿⠾⣥⠈⠻⣦⣤⡀⠀⠀⠀⠀⠀⠀⠈⠉⠉⠉⠉⠛⠛⠓⠒⠶⠶⠶⠼⢯⠹⣦⡀⠀⠀⠈⠉⠙⠛⠓⠶⠦⎥
⎢⠭⠨⢽⡇⠀⠻⣿⣿⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠨⢽⡆⢿⣷⡀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠨⠍⡇⠀⠀⠈⠻⣿⣿⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⠽⡇⠈⢿⣷⡀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣒⢸⡇⣿⠀⠀⠀⠀⠈⠻⣿⣿⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢐⣿⡇⠀⠈⢿⣷⡀⠀⠀⠀⠀⠀⠀⎥
⎢⣒⢐⡂⢻⠀⠀⠀⠀⠀⠀⠈⠻⣿⣿⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢐⣒⡇⠀⠀⠈⢿⣷⡀⠀⠀⠀⠀⠀⎥
⎢⠶⢐⡂⢸⡆⠀⠀⠀⠀⠀⠀⠀⠈⠻⣿⣿⢦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠰⣒⣧⠀⠀⠀⠈⢿⣷⡀⠀⠀⠀⠀⎥
⎢⠭⠰⠆⠸⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠳⣿⣿⣦⡀⠀⠀⠀⠀⠀⠀⠨⠆⣿⠀⠀⠀⠀⠈⢿⣷⡀⠀⠀⠀⎥
⎢⠭⠨⠅⠀⣧⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣿⣿⣦⡀⠀⠀⠀⠀⠨⠅⢸⠀⠀⠀⠀⠀⠈⢿⣷⡀⠀⠀⎥
⎢⢩⡭⠀⠀⢿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣿⣿⣦⡀⠀⠀⢨⠅⢸⠀⠀⠀⠀⠀⠀⠈⢿⣷⡀⠀⎥
⎢⢐⣺⠀⠀⢸⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣿⣿⣦⡀⢐⡇⢸⠀⠀⠀⠀⠀⠀⠀⠈⢿⣷⡀⎥
⎢⣒⣖⠀⠀⢸⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣿⣿⣖⡆⢸⡇⠀⠀⠀⠀⠀⠀⠀⠈⢿⣧⎥
⎢⣷⢿⣶⣶⡶⣇⣆⣆⣖⡖⣴⣴⢰⢰⢰⢢⠢⠆⠆⠆⠆⠖⠴⠴⠸⠽⢿⣷⡾⣷⣶⣶⣶⣶⡶⠶⠶⠶⠾⠿⎥
⎢⣿⢾⣯⡝⠳⣦⣬⣍⡉⠉⠉⠉⠉⠉⠉⠛⠛⠛⠒⠒⠒⠒⠒⠒⠶⠶⢾⣯⡻⣮⣍⠉⠉⠉⠛⠓⠒⠒⠲⠶⎥
⎢⣿⢸⡟⣇⠀⠈⠙⠻⢿⣷⣦⣄⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⣿⡇⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⢸⡇⢻⡀⠀⠀⠀⠀⠈⠙⠻⢿⣷⣦⣄⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⣿⡇⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⎥
⎢⣿⢸⡇⠸⣇⠀⠀⠀⠀⠀⠀⠀⠀⠈⠙⠻⢿⣷⣦⣄⡀⠀⠀⠀⠀⠀⢸⡏⢿⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⎥
⎢⢻⣾⠁⠀⢿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠙⠻⢿⣷⣦⣄⡀⠀⢸⡇⢸⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⎥
⎣⡾⣿⠀⠀⠸⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠙⠻⠿⣷⣾⡇⢸⡆⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⎦

## Solution

The coupled velocity–pressure system is solved simultaneously using the
assembled block matrix.

In [11]:
v, p = solveField(K, F, support=[pres1, suppT, suppB]);

## Visualization

The computed velocity and pressure fields are visualized using the Gmsh
post-processing interface.

In [12]:
showDoFResults(v, name="v", visible=true)
showDoFResults(p, name="p");

In [13]:
openPostProcessor()

XOpenIM() failed
Fontconfig warning: using without calling FcInit()
